# 05 — Ensembles Base

Este notebook treina e otimiza os **classificadores de ensemble** do TCC nas 4 representações textuais.

## Ideia central dos Ensembles

Em vez de depender de um único modelo, ensembles **combinam múltiplos classificadores** para obter uma previsão mais robusta e precisa. Há duas estratégias principais:

| Estratégia | Como funciona | Reduz | Modelos |
|---|---|---|---|
| **Bagging** | Treina N modelos em paralelo em subamostras aleatórias do treino | Variância | Random Forest |
| **Boosting** | Treina N modelos em sequência, cada um corrigindo os erros do anterior | Bias | AdaBoost, Gradient Boosting, XGBoost |

## Classificadores abordados:
1. **Random Forest** — Bagging de múltiplas Árvores de Decisão
2. **AdaBoost** — Boosting adaptativo por reponderação de amostras
3. **Gradient Boosting** — Boosting por descida de gradiente
4. **XGBoost** — Boosting por gradiente otimizado com regularização

> **Pré-requisito:** Execute o Notebook 03 para gerar os splits em `data/processed/`.

## 1. Importações e Carregamento dos Dados

In [19]:
import sys
sys.path.append('..')  # Permite importar módulos da pasta src/ a partir de notebooks/

import os
import joblib
import numpy as np
import pandas as pd

from src.models import (
    criar_random_forest,         # Fábrica: RandomForestClassifier configurado com n_jobs=-1
    criar_adaboost,              # Fábrica: AdaBoostClassifier com algoritmo SAMME
    criar_gradient_boosting,     # Fábrica: GradientBoostingClassifier sklearn
    criar_xgboost,               # Fábrica: XGBClassifier com suporte a matrizes esparsas
    PARAMS_RANDOM_FOREST,        # Grid: n_estimators, max_depth, min_samples_split, max_features
    PARAMS_ADABOOST,             # Grid: n_estimators, learning_rate
    PARAMS_GRADIENT_BOOSTING,    # Grid: n_estimators, learning_rate, max_depth
    PARAMS_XGBOOST,              # Grid: n_estimators, max_depth, learning_rate, subsample
    otimizar_modelo,             # GridSearchCV com StratifiedKFold(5) e f1_weighted
    salvar_modelo,               # Persiste o best_estimator_ em disco (.joblib)
    para_denso,                  # Converte matriz esparsa para numpy array denso
)

os.makedirs('../results/metrics', exist_ok=True)

print('✅ Importações concluídas com sucesso!')

✅ Importações concluídas com sucesso!


In [20]:
# Carrega os 4 splits gerados no Notebook 03.
# As labels y são as mesmas em todos os splits — apenas as features X mudam.
X_train_bow,   X_test_bow,   y_train, y_test = joblib.load('../data/processed/splits_bow.joblib')
X_train_tfidf, X_test_tfidf, _,       _      = joblib.load('../data/processed/splits_tfidf.joblib')
X_train_w2v,   X_test_w2v,   _,       _      = joblib.load('../data/processed/splits_w2v.joblib')
X_train_glove, X_test_glove, _,       _      = joblib.load('../data/processed/splits_glove.joblib')

print(f'BoW    — Treino: {X_train_bow.shape}   | Teste: {X_test_bow.shape}')
print(f'TF-IDF — Treino: {X_train_tfidf.shape} | Teste: {X_test_tfidf.shape}')
print(f'W2V    — Treino: {X_train_w2v.shape}   | Teste: {X_test_w2v.shape}')
print(f'GloVe  — Treino: {X_train_glove.shape} | Teste: {X_test_glove.shape}')
print(f'\nDistribuição das classes no treino (0=sem ideação | 1=com ideação):')
print(y_train.value_counts().to_string())

BoW    — Treino: (3021, 3875)   | Teste: (756, 3875)
TF-IDF — Treino: (3021, 5000) | Teste: (756, 5000)
W2V    — Treino: (3021, 100)   | Teste: (756, 100)
GloVe  — Treino: (3021, 100) | Teste: (756, 100)

Distribuição das classes no treino (0=sem ideação | 1=com ideação):
label
0    2149
1     872


## 2. Random Forest

O **Random Forest** é um ensemble de Bagging que treina N Árvores de Decisão **independentes** em paralelo, cada uma em uma subamostra aleatória do dataset (bootstrap), e combina suas predições por **votação majoritária**.

```
Dataset de treino
      │
      ├── Bootstrap 1 → Árvore 1 → predição 1
      ├── Bootstrap 2 → Árvore 2 → predição 2  →  Votação → classe final
      ├── Bootstrap 3 → Árvore 3 → predição 3
      └── ...         → Árvore N → predição N
```

**Dois tipos de aleatoriedade** tornam o RF robusto ao overfitting:
1. **Bagging**: cada árvore treina em uma amostra com reposição (~63% dos dados originais)
2. **Feature Sampling**: em cada divisão de nó, apenas `max_features` features são consideradas — as árvores ficam *decorrelacionadas* entre si

```python
PARAMS_RANDOM_FOREST = {
    'n_estimators'     : [100, 200, 300],   # Quantidade de árvores no ensemble
    'max_depth'        : [None, 10, 20],    # Profundidade máxima de cada árvore
    'min_samples_split': [2, 5],            # Mínimo de amostras para dividir um nó
    'max_features'     : ['sqrt', 'log2'],  # Features por split: sqrt(n) ou log2(n)
}
# Total: 3 × 3 × 2 × 2 = 36 combinações × 5 folds = 180 treinos
```

### 2.1 Random Forest × BoW

In [21]:
print('=' * 55)
print('  Random Forest × BoW')
print('=' * 55)

# n_jobs=-1 dentro do RF usa todos os núcleos para treinar as árvores em paralelo.
rf_bow = criar_random_forest()
grid_rf_bow = otimizar_modelo(rf_bow, PARAMS_RANDOM_FOREST, X_train_bow, y_train)
salvar_modelo(grid_rf_bow.best_estimator_, '../results/metrics/rf_bow_best.joblib')

  Random Forest × BoW
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Melhores parâmetros : {'max_depth': None, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 100}
Melhor score (f1_weighted): 0.8705
Modelo salvo em: ../results/metrics/rf_bow_best.joblib


### 2.2 Random Forest × TF-IDF

In [22]:
print('=' * 55)
print('  Random Forest × TF-IDF')
print('=' * 55)

rf_tfidf = criar_random_forest()
grid_rf_tfidf = otimizar_modelo(rf_tfidf, PARAMS_RANDOM_FOREST, X_train_tfidf, y_train)
salvar_modelo(grid_rf_tfidf.best_estimator_, '../results/metrics/rf_tfidf_best.joblib')

  Random Forest × TF-IDF
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Melhores parâmetros : {'max_depth': None, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}
Melhor score (f1_weighted): 0.8784
Modelo salvo em: ../results/metrics/rf_tfidf_best.joblib


### 2.3 Random Forest × Word2Vec

In [23]:
print('=' * 55)
print('  Random Forest × Word2Vec')
print('=' * 55)

rf_w2v = criar_random_forest()
grid_rf_w2v = otimizar_modelo(rf_w2v, PARAMS_RANDOM_FOREST, para_denso(X_train_w2v), y_train)
salvar_modelo(grid_rf_w2v.best_estimator_, '../results/metrics/rf_w2v_best.joblib')

  Random Forest × Word2Vec
Fitting 5 folds for each of 36 candidates, totalling 180 fits



Melhores parâmetros : {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 300}
Melhor score (f1_weighted): 0.8532
Modelo salvo em: ../results/metrics/rf_w2v_best.joblib


### 2.4 Random Forest × GloVe

In [24]:
print('=' * 55)
print('  Random Forest × GloVe')
print('=' * 55)

rf_glove = criar_random_forest()
grid_rf_glove = otimizar_modelo(rf_glove, PARAMS_RANDOM_FOREST, para_denso(X_train_glove), y_train)
salvar_modelo(grid_rf_glove.best_estimator_, '../results/metrics/rf_glove_best.joblib')

  Random Forest × GloVe
Fitting 5 folds for each of 36 candidates, totalling 180 fits



Melhores parâmetros : {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 300}
Melhor score (f1_weighted): 0.8477
Modelo salvo em: ../results/metrics/rf_glove_best.joblib


## 3. Resumo dos Resultados — Random Forest

In [25]:
resultados_rf = pd.DataFrame([
    {'Modelo': 'Random Forest', 'Representação': 'BoW',      'Melhores Params': grid_rf_bow.best_params_,   'F1-Score (CV)': round(grid_rf_bow.best_score_, 4)},
    {'Modelo': 'Random Forest', 'Representação': 'TF-IDF',   'Melhores Params': grid_rf_tfidf.best_params_, 'F1-Score (CV)': round(grid_rf_tfidf.best_score_, 4)},
    {'Modelo': 'Random Forest', 'Representação': 'Word2Vec', 'Melhores Params': grid_rf_w2v.best_params_,   'F1-Score (CV)': round(grid_rf_w2v.best_score_, 4)},
    {'Modelo': 'Random Forest', 'Representação': 'GloVe',    'Melhores Params': grid_rf_glove.best_params_,  'F1-Score (CV)': round(grid_rf_glove.best_score_, 4)},
])
display(resultados_rf)
resultados_rf.to_csv('../results/metrics/resultados_rf.csv', index=False)
print('✅ resultados_rf.csv salvo em results/metrics/')

,Modelo,Representação,Melhores Params,F1-Score (CV)
0,Random Forest,BoW,"{'max_depth': None, 'max_features': 'log2', 'm...",0.8705
1,Random Forest,TF-IDF,"{'max_depth': None, 'max_features': 'log2', 'm...",0.8784
2,Random Forest,Word2Vec,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.8532
3,Random Forest,GloVe,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.8477


✅ resultados_rf.csv salvo em results/metrics/


## 4. AdaBoost

O **AdaBoost** treina classificadores fracos (**weak learners**, Decision Stumps — árvores com `max_depth=1`) em **sequência**, aumentando progressivamente o peso das amostras classificadas errado.

```python
PARAMS_ADABOOST = {
    'n_estimators' : [50, 100, 200],      # Número de stumps (iterações)
    'learning_rate': [0.5, 1.0, 1.5],     # Peso de cada stump na predição final
}
# Total: 3 × 3 = 9 combinações × 5 folds = 45 treinos
```

### 4.1 AdaBoost × BoW

In [26]:
print('=' * 55)
print('  AdaBoost × BoW')
print('=' * 55)

ada_bow = criar_adaboost()
grid_ada_bow = otimizar_modelo(ada_bow, PARAMS_ADABOOST, X_train_bow, y_train)
salvar_modelo(grid_ada_bow.best_estimator_, '../results/metrics/ada_bow_best.joblib')

  AdaBoost × BoW
Fitting 5 folds for each of 9 candidates, totalling 45 fits

Melhores parâmetros : {'learning_rate': 1.5, 'n_estimators': 200}
Melhor score (f1_weighted): 0.8048
Modelo salvo em: ../results/metrics/ada_bow_best.joblib


### 4.2 AdaBoost × TF-IDF

In [27]:
print('=' * 55)
print('  AdaBoost × TF-IDF')
print('=' * 55)

ada_tfidf = criar_adaboost()
grid_ada_tfidf = otimizar_modelo(ada_tfidf, PARAMS_ADABOOST, X_train_tfidf, y_train)
salvar_modelo(grid_ada_tfidf.best_estimator_, '../results/metrics/ada_tfidf_best.joblib')

  AdaBoost × TF-IDF
Fitting 5 folds for each of 9 candidates, totalling 45 fits

Melhores parâmetros : {'learning_rate': 1.5, 'n_estimators': 200}
Melhor score (f1_weighted): 0.8340
Modelo salvo em: ../results/metrics/ada_tfidf_best.joblib


### 4.3 AdaBoost × Word2Vec

In [28]:
print('=' * 55)
print('  AdaBoost × Word2Vec')
print('=' * 55)

ada_w2v = criar_adaboost()
grid_ada_w2v = otimizar_modelo(ada_w2v, PARAMS_ADABOOST, para_denso(X_train_w2v), y_train)
salvar_modelo(grid_ada_w2v.best_estimator_, '../results/metrics/ada_w2v_best.joblib')

  AdaBoost × Word2Vec
Fitting 5 folds for each of 9 candidates, totalling 45 fits

Melhores parâmetros : {'learning_rate': 1.5, 'n_estimators': 200}
Melhor score (f1_weighted): 0.8265
Modelo salvo em: ../results/metrics/ada_w2v_best.joblib


### 4.4 AdaBoost × GloVe

In [29]:
print('=' * 55)
print('  AdaBoost × GloVe')
print('=' * 55)

ada_glove = criar_adaboost()
grid_ada_glove = otimizar_modelo(ada_glove, PARAMS_ADABOOST, para_denso(X_train_glove), y_train)
salvar_modelo(grid_ada_glove.best_estimator_, '../results/metrics/ada_glove_best.joblib')

  AdaBoost × GloVe
Fitting 5 folds for each of 9 candidates, totalling 45 fits

Melhores parâmetros : {'learning_rate': 1.0, 'n_estimators': 200}
Melhor score (f1_weighted): 0.8021
Modelo salvo em: ../results/metrics/ada_glove_best.joblib


## 5. Resumo dos Resultados — AdaBoost

In [30]:
resultados_ada = pd.DataFrame([
    {'Modelo': 'AdaBoost', 'Representação': 'BoW',      'Melhores Params': grid_ada_bow.best_params_,   'F1-Score (CV)': round(grid_ada_bow.best_score_, 4)},
    {'Modelo': 'AdaBoost', 'Representação': 'TF-IDF',   'Melhores Params': grid_ada_tfidf.best_params_, 'F1-Score (CV)': round(grid_ada_tfidf.best_score_, 4)},
    {'Modelo': 'AdaBoost', 'Representação': 'Word2Vec', 'Melhores Params': grid_ada_w2v.best_params_,   'F1-Score (CV)': round(grid_ada_w2v.best_score_, 4)},
    {'Modelo': 'AdaBoost', 'Representação': 'GloVe',    'Melhores Params': grid_ada_glove.best_params_,  'F1-Score (CV)': round(grid_ada_glove.best_score_, 4)},
])
display(resultados_ada)
resultados_ada.to_csv('../results/metrics/resultados_ada.csv', index=False)
print('✅ resultados_ada.csv salvo em results/metrics/')

,Modelo,Representação,Melhores Params,F1-Score (CV)
0,AdaBoost,BoW,"{'learning_rate': 1.5, 'n_estimators': 200}",0.8048
1,AdaBoost,TF-IDF,"{'learning_rate': 1.5, 'n_estimators': 200}",0.8340
2,AdaBoost,Word2Vec,"{'learning_rate': 1.5, 'n_estimators': 200}",0.8265
3,AdaBoost,GloVe,"{'learning_rate': 1.0, 'n_estimators': 200}",0.8021


✅ resultados_ada.csv salvo em results/metrics/


## 6. Gradient Boosting

O **Gradient Boosting** treina cada nova árvore para prever os **resíduos** do conjunto anterior via descida de gradiente sobre *logloss*.

```python
PARAMS_GRADIENT_BOOSTING = {
    'n_estimators' : [100, 200],         # Quantidade de árvores sequenciais
    'learning_rate': [0.05, 0.1, 0.2],   # Taxa de aprendizado (shrinkage)
    'max_depth'    : [3, 5],             # Profundidade de cada árvore base
}
# Total: 2 × 3 × 2 = 12 combinações × 5 folds = 60 treinos
```

### 6.1 Gradient Boosting × BoW

In [31]:
print('=' * 55)
print('  Gradient Boosting × BoW')
print('=' * 55)

# O treino é sequencial — o paralelismo ocorre no nível dos 5 folds do GridSearchCV.
gb_bow = criar_gradient_boosting()
grid_gb_bow = otimizar_modelo(gb_bow, PARAMS_GRADIENT_BOOSTING, X_train_bow, y_train)
salvar_modelo(grid_gb_bow.best_estimator_, '../results/metrics/gb_bow_best.joblib')

  Gradient Boosting × BoW
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Melhores parâmetros : {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 200}
Melhor score (f1_weighted): 0.8509
Modelo salvo em: ../results/metrics/gb_bow_best.joblib


### 6.2 Gradient Boosting × TF-IDF

In [32]:
print('=' * 55)
print('  Gradient Boosting × TF-IDF')
print('=' * 55)

gb_tfidf = criar_gradient_boosting()
grid_gb_tfidf = otimizar_modelo(gb_tfidf, PARAMS_GRADIENT_BOOSTING, X_train_tfidf, y_train)
salvar_modelo(grid_gb_tfidf.best_estimator_, '../results/metrics/gb_tfidf_best.joblib')

  Gradient Boosting × TF-IDF
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Melhores parâmetros : {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 200}
Melhor score (f1_weighted): 0.8657
Modelo salvo em: ../results/metrics/gb_tfidf_best.joblib


### 6.3 Gradient Boosting × Word2Vec

In [33]:
print('=' * 55)
print('  Gradient Boosting × Word2Vec')
print('=' * 55)

gb_w2v = criar_gradient_boosting()
grid_gb_w2v = otimizar_modelo(gb_w2v, PARAMS_GRADIENT_BOOSTING, para_denso(X_train_w2v), y_train)
salvar_modelo(grid_gb_w2v.best_estimator_, '../results/metrics/gb_w2v_best.joblib')

  Gradient Boosting × Word2Vec
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Melhores parâmetros : {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 200}
Melhor score (f1_weighted): 0.8561
Modelo salvo em: ../results/metrics/gb_w2v_best.joblib


### 6.4 Gradient Boosting × GloVe

In [34]:
print('=' * 55)
print('  Gradient Boosting × GloVe')
print('=' * 55)

gb_glove = criar_gradient_boosting()
grid_gb_glove = otimizar_modelo(gb_glove, PARAMS_GRADIENT_BOOSTING, para_denso(X_train_glove), y_train)
salvar_modelo(grid_gb_glove.best_estimator_, '../results/metrics/gb_glove_best.joblib')

  Gradient Boosting × GloVe
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Melhores parâmetros : {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 200}
Melhor score (f1_weighted): 0.8478
Modelo salvo em: ../results/metrics/gb_glove_best.joblib


## 7. Resumo dos Resultados — Gradient Boosting

In [35]:
resultados_gb = pd.DataFrame([
    {'Modelo': 'Gradient Boosting', 'Representação': 'BoW',      'Melhores Params': grid_gb_bow.best_params_,   'F1-Score (CV)': round(grid_gb_bow.best_score_, 4)},
    {'Modelo': 'Gradient Boosting', 'Representação': 'TF-IDF',   'Melhores Params': grid_gb_tfidf.best_params_, 'F1-Score (CV)': round(grid_gb_tfidf.best_score_, 4)},
    {'Modelo': 'Gradient Boosting', 'Representação': 'Word2Vec', 'Melhores Params': grid_gb_w2v.best_params_,   'F1-Score (CV)': round(grid_gb_w2v.best_score_, 4)},
    {'Modelo': 'Gradient Boosting', 'Representação': 'GloVe',    'Melhores Params': grid_gb_glove.best_params_,  'F1-Score (CV)': round(grid_gb_glove.best_score_, 4)},
])
display(resultados_gb)
resultados_gb.to_csv('../results/metrics/resultados_gb.csv', index=False)
print('✅ resultados_gb.csv salvo em results/metrics/')

,Modelo,Representação,Melhores Params,F1-Score (CV)
0,Gradient Boosting,BoW,"{'learning_rate': 0.2, 'max_depth': 5, 'n_esti...",0.8509
1,Gradient Boosting,TF-IDF,"{'learning_rate': 0.2, 'max_depth': 5, 'n_esti...",0.8657
2,Gradient Boosting,Word2Vec,"{'learning_rate': 0.2, 'max_depth': 5, 'n_esti...",0.8561
3,Gradient Boosting,GloVe,"{'learning_rate': 0.2, 'max_depth': 5, 'n_esti...",0.8478


✅ resultados_gb.csv salvo em results/metrics/


## 8. XGBoost

O **XGBoost** (eXtreme Gradient Boosting) é uma implementação altamente otimizada do Gradient Boosting com importantes avanços técnicos:

```
Gradient Boosting clássico
         +
Regularização L1 (Lasso) e L2 (Ridge) → penaliza modelos muito complexos
         +
Algoritmo de split por histograma → muito mais rápido que força bruta
         +
Suporte nativo a matrizes esparsas → ideal para BoW/TF-IDF
         +
Paralelismo intra-árvore → usa n_jobs=-1 dentro de cada árvore
         =
XGBoost
```

**Por que o XGBoost tende a superar o Gradient Boosting clássico?**
- **Regularização embutida** (`reg_alpha`, `reg_lambda`): evita overfitting mesmo com mais árvores
- **`subsample`**: amostragem estocástica de linhas por árvore (como no Bagging) reduz variância
- **Velocidade**: o algoritmo de split por histograma é até 10× mais rápido que o Gradient Boosting do scikit-learn para datasets médios

**Hiperparâmetros otimizados via GridSearchCV (5-Fold Estratificado):**
```python
PARAMS_XGBOOST = {
    'n_estimators' : [100, 200],          # Quantidade de árvores
    'max_depth'    : [4, 6, 8],           # Profundidade máxima de cada árvore
    'learning_rate': [0.05, 0.1, 0.2],    # Taxa de aprendizado (shrinkage)
    'subsample'    : [0.8, 1.0],          # Fração de amostras por árvore
}
# Total: 2 × 3 × 3 × 2 = 36 combinações × 5 folds = 180 treinos
```

### 8.1 XGBoost × BoW

In [36]:
print('=' * 55)
print('  XGBoost × BoW')
print('=' * 55)

# XGBoost aceita nativamente matrizes esparsas (scipy.sparse) sem precisar de para_denso().
# Isso é uma vantagem importante sobre o GradientBoostingClassifier do sklearn,
# que converte internamente para denso e usa mais memória.
# n_jobs=-1 dentro do XGBClassifier paraleliza os splits intra-árvore.
xgb_bow = criar_xgboost()
grid_xgb_bow = otimizar_modelo(xgb_bow, PARAMS_XGBOOST, X_train_bow, y_train)
salvar_modelo(grid_xgb_bow.best_estimator_, '../results/metrics/xgb_bow_best.joblib')

  XGBoost × BoW
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Melhores parâmetros : {'learning_rate': 0.2, 'max_depth': 8, 'n_estimators': 200, 'subsample': 1.0}
Melhor score (f1_weighted): 0.8492
Modelo salvo em: ../results/metrics/xgb_bow_best.joblib


### 8.2 XGBoost × TF-IDF

In [37]:
print('=' * 55)
print('  XGBoost × TF-IDF')
print('=' * 55)

# TF-IDF de 5.000 features esparsa: o XGBoost aproveita a esparsidade nativamente.
# O subsample=0.8 faz com que cada árvore use apenas 80% das amostras de treino,
# introduzindo aleatoriedade que reduz overfitting (similar ao Bagging).
xgb_tfidf = criar_xgboost()
grid_xgb_tfidf = otimizar_modelo(xgb_tfidf, PARAMS_XGBOOST, X_train_tfidf, y_train)
salvar_modelo(grid_xgb_tfidf.best_estimator_, '../results/metrics/xgb_tfidf_best.joblib')

  XGBoost × TF-IDF
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Melhores parâmetros : {'learning_rate': 0.2, 'max_depth': 8, 'n_estimators': 200, 'subsample': 1.0}
Melhor score (f1_weighted): 0.8668
Modelo salvo em: ../results/metrics/xgb_tfidf_best.joblib


### 8.3 XGBoost × Word2Vec

In [38]:
print('=' * 55)
print('  XGBoost × Word2Vec')
print('=' * 55)

# Com 100 features densas, o XGBoost com max_depth=6 a 8 consegue capturar
# interações de ordem alta entre dimensões semânticas do embedding.
# A regularização L1/L2 embutida controla a complexidade nesse espaço denso.
xgb_w2v = criar_xgboost()
grid_xgb_w2v = otimizar_modelo(xgb_w2v, PARAMS_XGBOOST, para_denso(X_train_w2v), y_train)
salvar_modelo(grid_xgb_w2v.best_estimator_, '../results/metrics/xgb_w2v_best.joblib')

  XGBoost × Word2Vec
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Melhores parâmetros : {'learning_rate': 0.2, 'max_depth': 4, 'n_estimators': 100, 'subsample': 1.0}
Melhor score (f1_weighted): 0.8625
Modelo salvo em: ../results/metrics/xgb_w2v_best.joblib


### 8.4 XGBoost × GloVe

In [39]:
print('=' * 55)
print('  XGBoost × GloVe')
print('=' * 55)

# Embeddings GloVe pré-treinados NILC-USP (100d).
# Esperamos que esta seja a combinação de melhor desempenho entre os ensembles
# com embeddings, dado que GloVe supera W2V em quase todos os modelos anteriores.
xgb_glove = criar_xgboost()
grid_xgb_glove = otimizar_modelo(xgb_glove, PARAMS_XGBOOST, para_denso(X_train_glove), y_train)
salvar_modelo(grid_xgb_glove.best_estimator_, '../results/metrics/xgb_glove_best.joblib')

  XGBoost × GloVe
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Melhores parâmetros : {'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 200, 'subsample': 0.8}
Melhor score (f1_weighted): 0.8446
Modelo salvo em: ../results/metrics/xgb_glove_best.joblib


## 9. Resumo dos Resultados — XGBoost

In [40]:
resultados_xgb = pd.DataFrame([
    {'Modelo': 'XGBoost', 'Representação': 'BoW',      'Melhores Params': grid_xgb_bow.best_params_,   'F1-Score (CV)': round(grid_xgb_bow.best_score_, 4)},
    {'Modelo': 'XGBoost', 'Representação': 'TF-IDF',   'Melhores Params': grid_xgb_tfidf.best_params_, 'F1-Score (CV)': round(grid_xgb_tfidf.best_score_, 4)},
    {'Modelo': 'XGBoost', 'Representação': 'Word2Vec', 'Melhores Params': grid_xgb_w2v.best_params_,   'F1-Score (CV)': round(grid_xgb_w2v.best_score_, 4)},
    {'Modelo': 'XGBoost', 'Representação': 'GloVe',    'Melhores Params': grid_xgb_glove.best_params_,  'F1-Score (CV)': round(grid_xgb_glove.best_score_, 4)},
])
display(resultados_xgb)
resultados_xgb.to_csv('../results/metrics/resultados_xgb.csv', index=False)
print('✅ resultados_xgb.csv salvo em results/metrics/')

,Modelo,Representação,Melhores Params,F1-Score (CV)
0,XGBoost,BoW,"{'learning_rate': 0.2, 'max_depth': 8, 'n_esti...",0.8492
1,XGBoost,TF-IDF,"{'learning_rate': 0.2, 'max_depth': 8, 'n_esti...",0.8668
2,XGBoost,Word2Vec,"{'learning_rate': 0.2, 'max_depth': 4, 'n_esti...",0.8625
3,XGBoost,GloVe,"{'learning_rate': 0.1, 'max_depth': 8, 'n_esti...",0.8446


✅ resultados_xgb.csv salvo em results/metrics/


## 10. Resumo Geral — Todos os Ensembles Base

In [41]:
# Consolida os resultados dos 4 ensembles base lendo os CSVs individuais.
# Funciona mesmo que as células acima não tenham sido executadas nesta sessão.
df_rf  = pd.read_csv('../results/metrics/resultados_rf.csv')
df_ada = pd.read_csv('../results/metrics/resultados_ada.csv')
df_gb  = pd.read_csv('../results/metrics/resultados_gb.csv')
df_xgb = pd.read_csv('../results/metrics/resultados_xgb.csv')

resumo_ensembles = pd.concat([df_rf, df_ada, df_gb, df_xgb], ignore_index=True)

# Ordena do melhor para o pior F1-Score
resumo_ensembles = resumo_ensembles.sort_values('F1-Score (CV)', ascending=False).reset_index(drop=True)

display(resumo_ensembles)

resumo_ensembles.to_csv('../results/metrics/resultados_ensembles.csv', index=False)
print('✅ resultados_ensembles.csv salvo em results/metrics/')
print(f'\nMelhor ensemble encontrado:')
print(f"  {resumo_ensembles.iloc[0]['Modelo']} × {resumo_ensembles.iloc[0]['Representação']}")
print(f"  F1-Score (CV): {resumo_ensembles.iloc[0]['F1-Score (CV)']}")

,Modelo,Representação,Melhores Params,F1-Score (CV)
0,Random Forest,TF-IDF,"{'max_depth': None, 'max_features': 'log2', 'm...",0.8784
1,Random Forest,BoW,"{'max_depth': None, 'max_features': 'log2', 'm...",0.8705
2,XGBoost,TF-IDF,"{'learning_rate': 0.2, 'max_depth': 8, 'n_esti...",0.8668
3,Gradient Boosting,TF-IDF,"{'learning_rate': 0.2, 'max_depth': 5, 'n_esti...",0.8657
4,XGBoost,Word2Vec,"{'learning_rate': 0.2, 'max_depth': 4, 'n_esti...",0.8625
5,Gradient Boosting,Word2Vec,"{'learning_rate': 0.2, 'max_depth': 5, 'n_esti...",0.8561
6,Random Forest,Word2Vec,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.8532
7,Gradient Boosting,BoW,"{'learning_rate': 0.2, 'max_depth': 5, 'n_esti...",0.8509
8,XGBoost,BoW,"{'learning_rate': 0.2, 'max_depth': 8, 'n_esti...",0.8492
9,Gradient Boosting,GloVe,"{'learning_rate': 0.2, 'max_depth': 5, 'n_esti...",0.8478


✅ resultados_ensembles.csv salvo em results/metrics/

Melhor ensemble encontrado:
  Random Forest × TF-IDF
  F1-Score (CV): 0.8784
